# 02 — Install Kubernetes with Kubespray

In this notebook we:
1. Configure Ansible to reach our 3 VMs via SSH
2. Run a pre-K8s playbook (firewall, Docker, swap)
3. Run **Kubespray** to install Kubernetes (~30–60 min)
4. Run a post-install playbook to configure kubectl and access dashboards

> **Step 3 might take between 30–60 minutes.** Start it and come back.


## Prerequisites

You need the **floating IP** from `01_provision.ipynb`. Replace `A.B.C.D` with it throughout this notebook.


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

## Step 1: Configure Ansible

Open the file `immich-iac/ansible/ansible.cfg` in the Jupyter file browser.  
Find the line with `A.B.C.D` and replace it with your actual floating IP from `01_provision.ipynb`.  
Save the file.

Then copy it to the correct location:


In [ ]:
# runs in Chameleon Jupyter environment
# After editing ansible.cfg with your real floating IP:
cp /work/immich-iac/ansible/ansible.cfg.template /work/immich-iac/ansible/ansible.cfg

# Now open /work/immich-iac/ansible/ansible.cfg in the file browser
# and replace A.B.C.D with your real floating IP, then save.
echo "Edit the file, then run the next cell."

In [ ]:
# runs in Chameleon Jupyter environment
# Confirm the floating IP is set (should NOT show A.B.C.D)
grep "ProxyCommand" /work/immich-iac/ansible/ansible.cfg

## Step 2: Verify connectivity to all 3 nodes

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

cd /work/immich-iac/ansible
ansible -i inventory.yml all -m ping

All 3 nodes should return `pong`. If any fail, wait 1 minute and retry — the VMs may still be booting.

## Step 3: Run pre-Kubernetes configuration

This playbook:
- Disables the host firewall (cloud security groups handle this instead)
- Installs Docker
- Disables swap (required for Kubernetes)


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

cd /work/immich-iac/ansible
ansible-playbook -i inventory.yml pre_k8s/pre_k8s_configure.yml

## Step 4: Install Kubernetes with Kubespray

Start it and go do something else.**

When it finishes, the `PLAY RECAP` at the bottom must show `failed=0`.  
If it shows any failures, just re-run the same cell — Kubespray is safe to re-run.


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local
export ANSIBLE_CONFIG=/work/immich-iac/ansible/ansible.cfg
export ANSIBLE_ROLES_PATH=roles

cd /work/immich-iac/ansible/k8s/kubespray
ansible-playbook -i ../inventory/mycluster --become --become-user=root ./cluster.yml

## Step 5: Run post-install playbook

This playbook:
- Configures `kubectl` for the `cc` user on node1
- Adjusts cluster networking for Chameleon stability
- Installs ArgoCD CLI and Argo Workflows (for future use)
- Prints the Kubernetes dashboard token and ArgoCD password


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

cd /work/immich-iac/ansible
ansible-playbook -i inventory.yml post_k8s/post_k8s_configure.yml

**Save the dashboard token and any passwords printed above.**

## Step 6: Access the Kubernetes dashboard (optional)

Run this in your **local terminal** (not Jupyter), replacing `A.B.C.D` with your floating IP:

```bash
ssh -L 8443:127.0.0.1:8443 -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

Then inside that SSH session:
```bash
kubectl port-forward -n kube-system svc/kubernetes-dashboard 8443:443
```

Open `https://127.0.0.1:8443` in your browser and paste in the dashboard token.

---
**Continue to `03_deploy_services.ipynb`**
